# 🔬 MODEL MAYHEM: PCB DEFECT DETECTION
### End-to-End Deep Learning Training, Evaluation & Submission Pipeline
**Dataset:** [Roboflow Universe - PCB Defect Detection v2](https://universe.roboflow.com/uthm-project-visionbased-pcb-defect-detection/pcb-defect-detection-9ewqw-duvuw/dataset/2)  
**Competition Target:** High-precision localization and classification of PCB micro-defects.

**Target Classes (6):**
1. `missing_hole`
2. `mouse_bite`
3. `open_circuit`
4. `short`
5. `spur`
6. `spurious_copper`

**Evaluation Metrics:**
- **Primary:** `mAP@50:90` (COCO standard mAP across IoU 0.50 to 0.90)
- **Tie-Breaker 1:** `mAP@50`
- **Tie-Breaker 2:** `Macro Recall` (Average recall across all 6 defect classes)


## 1. Environment & GPU Verification
Verify that GPU acceleration (NVIDIA T4, V100, A100, or L4) is active.


In [ ]:
# Check GPU allocation
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:      {torch.cuda.get_device_name(0)}")
    print(f"Device Memory:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ WARNING: Running on CPU. For fast training, enable GPU via Runtime > Change runtime type > T4 GPU.")


## 2. Install Dependencies
Install Ultralytics (YOLO11/YOLOv8), Roboflow SDK, OpenCV, and evaluation utilities.


In [ ]:
!pip install -q --upgrade ultralytics roboflow gdown opencv-python-headless pandas pyyaml matplotlib
import ultralytics
ultralytics.checks()


## 3. Dataset Download & Structure Verification
Download the official competition dataset (2,586 annotated PCB images) in YOLOv8 format from Roboflow Universe.


In [ ]:
import os
import sys
import zipfile
import shutil
from pathlib import Path

# ==============================================================================
# DATASET DOWNLOAD / UPLOAD OPTIONS
# ==============================================================================
# Option 1 (Default): Google Drive Dataset Folder ID
GDRIVE_FOLDER_ID = "1w1hSif-38ggqxWsuMmVvK9NcJP9_Kp1z"  # @param {type:"string"}

# Option 2: Enter your free Roboflow API Key (optional)
ROBOFLOW_API_KEY = ""  # @param {type:"string"}

# Option 3: Paste direct Roboflow curl download URL (optional)
ROBOFLOW_CURL_URL = ""  # @param {type:"string"}

dataset_dir = "dataset"
dataset_path = Path(dataset_dir)

def has_images(dir_path):
    p = Path(dir_path)
    if not p.exists():
        return False
    exts = {"*.jpg", "*.png", "*.jpeg", "*.bmp"}
    for ext in exts:
        if any(p.rglob(ext)):
            return True
    return False

if has_images(dataset_dir):
    print(f"✅ Existing dataset found at: {os.path.abspath(dataset_dir)}")
elif GDRIVE_FOLDER_ID.strip():
    print(f"📥 Downloading dataset from Google Drive folder: {GDRIVE_FOLDER_ID.strip()}...")
    !gdown --folder "https://drive.google.com/drive/folders/{GDRIVE_FOLDER_ID.strip()}" -O dataset_download --remaining-ok
    download_path = Path("dataset_download")
    if download_path.exists():
        zips = list(download_path.rglob("*.zip"))
        if zips:
            for zf in zips:
                print(f"📦 Extracting {zf.name} to {dataset_dir}...")
                with zipfile.ZipFile(zf, "r") as z:
                    z.extractall(dataset_dir)
        elif has_images(download_path):
            dataset_dir = "dataset_download"
            dataset_path = Path(dataset_dir)
        else:
            # Copy contents if folder hierarchy
            for item in download_path.iterdir():
                dest = dataset_path / item.name
                if item.is_dir() and not dest.exists():
                    shutil.copytree(item, dest)
                elif item.is_file() and not dest.exists():
                    dataset_path.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(item, dest)
elif ROBOFLOW_API_KEY.strip():
    print("📥 Connecting to Roboflow Universe using API key...")
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY.strip())
    project = rf.workspace("uthm-project-visionbased-pcb-defect-detection").project("pcb-defect-detection-9ewqw-duvuw")
    dataset = project.version(2).download("yolov8")
    dataset_dir = dataset.location
    dataset_path = Path(dataset_dir)
elif ROBOFLOW_CURL_URL.strip():
    print("📥 Downloading dataset via provided Roboflow URL...")
    !curl -L "{ROBOFLOW_CURL_URL.strip()}" > roboflow_pcb.zip
    !unzip -q -o roboflow_pcb.zip -d dataset/
else:
    uploaded_zips = list(Path(".").glob("*.zip")) + list(Path("/content").glob("*.zip"))
    uploaded_zips = [z for z in uploaded_zips if "bundle" not in z.name.lower()]
    if uploaded_zips:
        chosen_zip = uploaded_zips[0]
        print(f"📦 Found uploaded dataset archive: {chosen_zip}. Extracting to {dataset_dir}...")
        with zipfile.ZipFile(chosen_zip, "r") as z:
            z.extractall(dataset_dir)
    else:
        print("⚠️ Checking interactive upload...")
        try:
            from google.colab import files
            print("Select your downloaded pcb defect dataset zip file to upload:")
            uploaded = files.upload()
            for filename in uploaded.keys():
                if filename.endswith(".zip"):
                    print(f"Extracting {filename}...")
                    with zipfile.ZipFile(filename, "r") as z:
                        z.extractall(dataset_dir)
                    break
        except Exception as e:
            print(f"Interactive upload skipped: {e}")

# Check if images are actually present
if not has_images(dataset_dir):
    raise RuntimeError(
        "\n❌ DATASET NOT FOUND!\n"
        "Ultralytics cannot train because no images were found in the dataset directory.\n"
        "Please ensure the Google Drive folder ID is accessible or upload the dataset ZIP file."
    )

print(f"✅ Dataset successfully verified at: {os.path.abspath(dataset_dir)}")


### Fix and Normalize dataset.yaml
Ensure class order and names strictly match competition specifications.


In [ ]:
import yaml
from pathlib import Path
from collections import Counter

dataset_path = Path(dataset_dir).resolve()

# Canonical competition class mapping (6 classes)
CANONICAL_CLASSES = [
    "missing_hole",
    "mouse_bite",
    "open_circuit",
    "short",
    "spur",
    "spurious_copper"
]

# Robust split locator: search for train / valid / test directories
def find_split_dir(root_path, split_name):
    candidates = [
        root_path / split_name / "images",
        root_path / split_name,
    ]
    candidates.extend(list(root_path.glob(f"*/{split_name}/images")))
    candidates.extend(list(root_path.glob(f"*/{split_name}")))
    for c in candidates:
        if c.exists() and (any(c.glob("*.jpg")) or any(c.glob("*.png")) or any(c.glob("*.jpeg"))):
            return c
    return root_path / split_name

train_dir = find_split_dir(dataset_path, "train")
val_dir = find_split_dir(dataset_path, "valid")
if not val_dir.exists():
    val_dir = find_split_dir(dataset_path, "val")
test_dir = find_split_dir(dataset_path, "test")

# Create standardized dataset.yaml
data_yaml_dict = {
    "path": str(dataset_path).replace("\\", "/"),
    "train": str(train_dir.resolve().relative_to(dataset_path)).replace("\\", "/"),
    "val": str(val_dir.resolve().relative_to(dataset_path)).replace("\\", "/"),
    "test": str(test_dir.resolve().relative_to(dataset_path)).replace("\\", "/") if test_dir.exists() else None,
    "nc": len(CANONICAL_CLASSES),
    "names": {i: name for i, name in enumerate(CANONICAL_CLASSES)},
}
if data_yaml_dict["test"] is None:
    del data_yaml_dict["test"]

yaml_file = "dataset_colab.yaml"
with open(yaml_file, "w") as f:
    yaml.dump(data_yaml_dict, f, default_flow_style=False, sort_keys=False)

print(f"✅ Generated dataset configuration: {yaml_file}")
print(open(yaml_file).read())


## 4. Model Training: SOTA YOLO11x with PCB Micro-Defect Augmentations
### Hyperparameter Tuning for PCB Inspection:
- **Resolution (`imgsz=1024`):** Crucial for detecting microscopic spurs, pinholes, and mouse bites.
- **Augmentations:**
  - `mosaic=1.0`: Exposes network to multiple micro-defects at varying contexts.
  - `degrees=15.0`, `flipud=0.5`, `fliplr=0.5`: Robustness to arbitrary board orientations.
  - `scale=0.5`: Multi-scale defect variance.
  - `copy_paste=0.15`: Addresses rare defect instance distribution.
  - `close_mosaic=10`: Disables mosaic during the final 10 epochs for crisp bounding box boundary convergence.
- **Loss Gains (`box=7.5`):** Prioritizes precise coordinate regression to maximize high-threshold mAP (mAP@75, mAP@90).


In [ ]:
import os
import torch
from pathlib import Path
from ultralytics import YOLO

# Select architecture: YOLO11x (State of the Art) or YOLO11l
MODEL_NAME = "yolo11x.pt"  # Pretrained YOLO11x
EPOCHS = 50                # Increase to 100 for maximum score if time permits
IMGSZ = 1024               # High resolution for tiny defect detection
BATCH_SIZE = 8             # 8 for 16GB GPU (T4), 16 for A100/V100

print(f"Initializing YOLO model: {MODEL_NAME}")
model = YOLO(MODEL_NAME)

train_results = model.train(
    data=yaml_file,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH_SIZE,
    device=0 if torch.cuda.is_available() else "cpu",
    project="pcb_competition",
    name="yolo11x_pcb",
    optimizer="auto",
    lr0=0.01,
    lrf=0.01,
    warmup_epochs=3.0,
    cos_lr=True,
    # Defect-tailored augmentations:
    mosaic=1.0,
    mixup=0.15,
    degrees=15.0,
    flipud=0.5,
    fliplr=0.5,
    scale=0.5,
    copy_paste=0.15,
    close_mosaic=10,
    # High box gain for strict mAP@50:90:
    box=7.5,
    cls=0.5,
    dfl=1.5,
    plots=True,
    save=True,
)

# Reliably resolve best weights path
if hasattr(train_results, "save_dir") and (Path(train_results.save_dir) / "weights" / "best.pt").exists():
    best_weights_path = Path(train_results.save_dir) / "weights" / "best.pt"
else:
    candidates = sorted(Path(".").rglob("weights/best.pt"), key=os.path.getmtime, reverse=True)
    if candidates:
        best_weights_path = candidates[0]
    else:
        best_weights_path = Path("pcb_competition/yolo11x_pcb/weights/best.pt")

print(f"\n🎯 Training Finished! Best weights saved to: {best_weights_path.resolve()}")


## 4.1 Multi-Model Architecture Suite (5 Model Comparison)
To ensure the dataset is comprehensively trained and thoroughly benchmarked across distinct inductive biases, we support training 5 representative deep learning detection models:

| Model Architecture | Backbone Family | Framework | Key Strengths for PCB Defect Detection |
| :--- | :--- | :--- | :--- |
| **YOLO11x** | C3k2 + SPPF | 1-Stage SOTA | Highest mAP@50:90, ultra-fast, multi-scale feature pyramids |
| **YOLOv8x** | C2f + SPPF | 1-Stage Anchor-Free | Proven robustness, high recall on tiny traces & spurs |
| **RT-DETR-l / RT-DETR-x** | Hybrid Encoder + Transformer | Transformer Detector | End-to-end NMS-free, global receptive field for board-level defect context |
| **Faster R-CNN (ResNet-50 V2)** | ResNet-50-FPN-v2 | 2-Stage Region Proposal | Dense RPN proposals, high precision localization on micro-defects |
| **Faster R-CNN (MobileNet-V3 / V2)** | MobileNet-FPN | 2-Stage Lightweight | Efficient inverted residual blocks, low latency, edge deployment |

> **Note on Classification Backbones (MobileNet, EfficientNet, DenseNet, Inception, Xception, ResNet):**  
> In object detection tasks evaluated on bounding box `mAP@50:90`, classification backbones function as feature extractors within detection frameworks (such as Faster R-CNN with FPN or RetinaNet). YOLO11x and RT-DETR feature integrated multi-scale feature extractors specifically engineered for spatial coordinate regression.


In [ ]:
# Train Secondary Models for Multi-Model Comparison & Ensembling
# Choose which models to train: 'yolov8x.pt', 'rtdetr-l.pt', or torchvision Faster R-CNN with ResNet/MobileNet

TRAIN_MULTI_MODELS = False  # @param {"type":"boolean"}

if TRAIN_MULTI_MODELS:
    # 1. Train YOLOv8x
    print("\n" + "=" * 60)
    print("🚀 Training Model 2: YOLOv8x...")
    print("=" * 60)
    model_v8 = YOLO("yolov8x.pt")
    model_v8.train(
        data=yaml_file,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH_SIZE,
        device=0 if torch.cuda.is_available() else "cpu",
        project="pcb_competition",
        name="yolov8x_pcb",
        box=7.5,
        cls=0.5,
        dfl=1.5,
        mosaic=1.0,
        flipud=0.5,
        fliplr=0.5
    )

    # 2. Train RT-DETR (Real-Time Detection Transformer)
    print("\n" + "=" * 60)
    print("🚀 Training Model 3: RT-DETR-l (Transformer Detector)...")
    print("=" * 60)
    from ultralytics import RTDETR
    model_rtdetr = RTDETR("rtdetr-l.pt")
    model_rtdetr.train(
        data=yaml_file,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH_SIZE,
        device=0 if torch.cuda.is_available() else "cpu",
        project="pcb_competition",
        name="rtdetr_pcb"
    )

    # 3. Train Faster R-CNN (ResNet-50 V2 FPN) via train_torchvision.py
    print("\n" + "=" * 60)
    print("🚀 Training Model 4: Faster R-CNN with ResNet-50 V2 Backbone...")
    print("=" * 60)
    !python train_torchvision.py --model fasterrcnn_resnet50_v2 --data {yaml_file} --epochs 15 --batch 4 --imgsz 1024 --output-weights pcb_runs/fasterrcnn_resnet50.pth

    # 4. Train Faster R-CNN (MobileNet V3 FPN) via train_torchvision.py
    print("\n" + "=" * 60)
    print("🚀 Training Model 5: Faster R-CNN with MobileNet V3 Backbone...")
    print("=" * 60)
    !python train_torchvision.py --model fasterrcnn_mobilenet_v3 --data {yaml_file} --epochs 15 --batch 4 --imgsz 1024 --output-weights pcb_runs/fasterrcnn_mobilenet.pth

    # Note: To train with other user-suggested classification backbones (MobileNet-V2, EfficientNet-B0, DenseNet-121):
    # !python train_torchvision.py --model fasterrcnn_mobilenet_v2 --data {yaml_file} --epochs 15 --batch 4 --imgsz 1024 --output-weights pcb_runs/fasterrcnn_mobilenet_v2.pth
    # !python train_torchvision.py --model fasterrcnn_efficientnet_b0 --data {yaml_file} --epochs 15 --batch 4 --imgsz 1024 --output-weights pcb_runs/fasterrcnn_efficientnet.pth
    # !python train_torchvision.py --model fasterrcnn_densenet121 --data {yaml_file} --epochs 15 --batch 4 --imgsz 1024 --output-weights pcb_runs/fasterrcnn_densenet.pth
else:
    print("ℹ️ Single model training completed. Set TRAIN_MULTI_MODELS = True to train YOLOv8x, RT-DETR, Faster R-CNN ResNet50-V2, and MobileNet-V3.")


## 5. Official Competition Metric Evaluation
Compute **mAP@50:90 (Primary)**, **mAP@50 (Tie-Breaker 1)**, and **Macro Recall (Tie-Breaker 2)** on the validation split.


In [ ]:
import os
from pathlib import Path
from ultralytics import YOLO

# Dynamically locate trained weights if not set
if 'best_weights_path' not in locals() or not Path(best_weights_path).exists():
    candidates = sorted(Path(".").rglob("weights/best.pt"), key=os.path.getmtime, reverse=True)
    if candidates:
        best_weights_path = candidates[0]
        print(f"🔍 Located trained weights at: {best_weights_path}")
    else:
        raise FileNotFoundError(
            "\n❌ No trained weights found ('best.pt')!\n"
            "This error occurs because the training step (Section 4) did not finish successfully.\n"
            "Common causes:\n"
            "  1. Section 3 didn't find the dataset (no Roboflow API key or ZIP provided).\n"
            "  2. Section 4 encountered an error or was stopped early.\n\n"
            "Please scroll up to Section 3 to ensure the dataset is downloaded, then re-run Section 4!"
        )

best_model = YOLO(str(best_weights_path))

# 1. Standard Validation
print("=" * 60)
print("📊 STANDARD VALIDATION METRICS:")
print("=" * 60)
metrics = best_model.val(data=yaml_file, imgsz=IMGSZ, batch=BATCH_SIZE)

map50_90 = metrics.box.map
map50 = metrics.box.map50
macro_recall = metrics.box.mr
macro_precision = metrics.box.mp

print("\n🏆 OFFICIAL LEADERBOARD METRICS:")
print(f"  • Primary Metric (mAP@50:90): {map50_90:.4f}")
print(f"  • Tie-Breaker 1  (mAP@50):    {map50:.4f}")
print(f"  • Tie-Breaker 2  (Recall):    {macro_recall:.4f}")
print(f"  • Macro Precision:            {macro_precision:.4f}")

# 2. Test-Time Augmentation (TTA) Validation
print("\n" + "=" * 60)
print("⚡ TEST-TIME AUGMENTATION (TTA) VALIDATION:")
print("=" * 60)
metrics_tta = best_model.val(data=yaml_file, imgsz=IMGSZ, batch=BATCH_SIZE, augment=True)
print(f"  • TTA Primary (mAP@50:90):    {metrics_tta.box.map:.4f} (Delta: {metrics_tta.box.map - map50_90:+.4f})")
print(f"  • TTA Tie-Breaker (mAP@50):   {metrics_tta.box.map50:.4f} (Delta: {metrics_tta.box.map50 - map50:+.4f})")
print(f"  • TTA Macro Recall:           {metrics_tta.box.mr:.4f} (Delta: {metrics_tta.box.mr - macro_recall:+.4f})")


## 6. Test Inference & Submission CSV Generation
Generates predictions in the exact required schema:
```csv
image_name,class_name,x_min,y_min,x_max,y_max,confidence
```


In [ ]:
import cv2
import pandas as pd
from tqdm import tqdm

test_images_dir = test_dir if test_dir.exists() else val_dir
test_images = sorted([f for f in test_images_dir.glob("*") if f.suffix.lower() in [".jpg", ".png", ".jpeg"]])
print(f"Found {len(test_images)} test images in {test_images_dir}")

submission_rows = []
USE_TTA_INFERENCE = True  # Enable multi-scale TTA for boosted test score
CONF_THRESHOLD = 0.25     # High precision threshold
IOU_THRESHOLD = 0.50

for img_path in tqdm(test_images, desc="Predicting PCB defects"):
    results = best_model.predict(
        source=str(img_path),
        imgsz=IMGSZ,
        conf=CONF_THRESHOLD,
        iou=IOU_THRESHOLD,
        augment=USE_TTA_INFERENCE,
        verbose=False
    )
    res = results[0]
    orig_h, orig_w = res.orig_shape

    boxes = res.boxes
    if boxes is not None and len(boxes) > 0:
        xyxy = boxes.xyxy.cpu().numpy()
        confs = boxes.conf.cpu().numpy()
        classes = boxes.cls.cpu().numpy()

        for box, conf, cls_idx in zip(xyxy, confs, classes):
            x1, y1, x2, y2 = box
            # Clip bounds to image dimensions
            x1 = max(0.0, min(float(orig_w), float(x1)))
            y1 = max(0.0, min(float(orig_h), float(y1)))
            x2 = max(0.0, min(float(orig_w), float(x2)))
            y2 = max(0.0, min(float(orig_h), float(y2)))

            if x2 <= x1 or y2 <= y1:
                continue

            cls_name = CANONICAL_CLASSES[int(cls_idx) % len(CANONICAL_CLASSES)]
            submission_rows.append({
                "image_name": img_path.name,
                "class_name": cls_name,
                "x_min": round(x1, 2),
                "y_min": round(y1, 2),
                "x_max": round(x2, 2),
                "y_max": round(y2, 2),
                "confidence": round(float(conf), 4)
            })

submission_df = pd.DataFrame(submission_rows, columns=[
    "image_name", "class_name", "x_min", "y_min", "x_max", "y_max", "confidence"
])
submission_csv = "submission.csv"
submission_df.to_csv(submission_csv, index=False)
print(f"\n✅ Generated {submission_csv} with {len(submission_df)} defect detections.")


## 6.1 Multi-Model Ensemble with Weighted Box Fusion (WBF)
### Why Weighted Box Fusion (WBF) Boosts mAP@50:90:
- **Traditional NMS Flaw:** Standard NMS operates on a "winner-takes-all" principle — it selects one bounding box with the highest score and discards overlapping boxes. If that single box is even slightly misaligned, precision drops significantly at high IoU thresholds (`IoU=0.75` to `IoU=0.90`).
- **WBF Advantage:** Weighted Box Fusion clusters overlapping boxes from all models and computes a **confidence-weighted spatial average** of the coordinate boundaries:
  $$x_{1,\text{fused}} = \frac{\sum_i w_i \cdot c_i \cdot x_{1,i}}{\sum_i w_i \cdot c_i}$$
  This produces consensus bounding boxes that are substantially tighter and more accurate, directly maximizing the competition's primary metric: `mAP@50:90`.
- Furthermore, WBF confidence weighting naturally suppresses false positives that are only predicted by a single outlier model.


In [ ]:
# Execute Weighted Box Fusion (WBF) Ensemble across Model Checkpoints / TTA
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from ensemble import weighted_boxes_fusion, ensemble_csv_files
from inference import run_inference

print("=" * 60)
print("⚡ RUNNING WEIGHTED BOX FUSION (WBF) ENSEMBLE")
print("=" * 60)

# 1. Save standard YOLO11x predictions
submission_df.to_csv("submission_yolo11x.csv", index=False)
csv_ensemble_list = ["submission_yolo11x.csv"]
ensemble_weights = [1.2]

# 2. Generate TTA predictions for ensembling
print("\n--- Generating TTA predictions for primary model ---")
df_tta = run_inference(
    weights_path=str(best_weights),
    source_path=str(test_images_dir),
    output_csv="submission_tta.csv",
    conf_threshold=0.15,
    iou_threshold=0.50,
    imgsz=IMGSZ,
    use_tta=True,
)
csv_ensemble_list.append("submission_tta.csv")
ensemble_weights.append(1.0)

# 3. Add predictions from secondary models if they have been trained
candidates = [
    ("pcb_competition/yolov8x_pcb/weights/best.pt", "submission_yolov8x.csv", 1.0),
    ("pcb_competition/rtdetr_pcb/weights/best.pt", "submission_rtdetr.csv", 1.0),
    ("pcb_runs/fasterrcnn_resnet50.pth", "submission_fasterrcnn_resnet50.csv", 0.9),
    ("pcb_runs/fasterrcnn_mobilenet.pth", "submission_fasterrcnn_mobilenet.csv", 0.8),
    ("pcb_runs/fasterrcnn_mobilenet_v2.pth", "submission_fasterrcnn_mobilenet_v2.csv", 0.8),
    ("pcb_runs/fasterrcnn_efficientnet.pth", "submission_fasterrcnn_efficientnet.csv", 0.8),
    ("pcb_runs/fasterrcnn_densenet.pth", "submission_fasterrcnn_densenet.csv", 0.8),
    ("pcb_runs/torchvision_best.pth", "submission_torchvision.csv", 0.8),
]

for weight_file, out_csv, w in candidates:
    if Path(weight_file).exists() and out_csv not in csv_ensemble_list:
        print(f"\n--- Running inference for checkpoint: {weight_file} ---")
        run_inference(
            weights_path=weight_file,
            source_path=str(test_images_dir),
            output_csv=out_csv,
            conf_threshold=0.15,
            imgsz=IMGSZ,
        )
        csv_ensemble_list.append(out_csv)
        ensemble_weights.append(w)

print(f"\nEnsembling {len(csv_ensemble_list)} prediction sets: {csv_ensemble_list} with weights {ensemble_weights}")

ensemble_df = ensemble_csv_files(
    csv_paths=csv_ensemble_list,
    output_path="submission_ensemble.csv",
    weights=ensemble_weights,
    method="wbf",
    iou_thr=0.55,
    skip_box_thr=0.10,
    conf_type="avg",
)

# Overwrite submission.csv with the superior ensembled predictions
ensemble_df.to_csv("submission.csv", index=False)
print(f"\n🎉 WBF Ensemble complete! 'submission.csv' updated with {len(ensemble_df)} fused detections.")


## 7. Submission Integrity Verification
Validate that the CSV strictly adheres to all competition rules before submission.


In [ ]:
# Strict Verification Routine
df_sub = pd.read_csv("submission.csv")

assert list(df_sub.columns) == ["image_name", "class_name", "x_min", "y_min", "x_max", "y_max", "confidence"], "Column names mismatch!"
assert set(df_sub["class_name"].unique()).issubset(set(CANONICAL_CLASSES)), f"Unknown classes found: {set(df_sub['class_name'].unique()) - set(CANONICAL_CLASSES)}"
assert (df_sub["confidence"] >= 0.0).all() and (df_sub["confidence"] <= 1.0).all(), "Confidence scores outside [0, 1]!"
assert (df_sub["x_max"] > df_sub["x_min"]).all(), "Invalid x coordinates (x_max <= x_min)!"
assert (df_sub["y_max"] > df_sub["y_min"]).all(), "Invalid y coordinates (y_max <= y_min)!"
assert (df_sub["x_min"] >= 0).all() and (df_sub["y_min"] >= 0).all(), "Negative coordinates found!"

print("🎉 VERIFICATION PASSED: Submission CSV is 100% compliant with competition requirements!")
print("\nFirst 10 sample predictions:")
display(df_sub.head(10))
print("\nClass detection summary:")
print(df_sub["class_name"].value_counts())


## 8. Export Submission Bundle
Package trained weights (`best.pt`), `submission.csv`, and inference code into a zip archive for one-click download.


In [ ]:
import zipfile
import shutil
from pathlib import Path

zip_filename = "pcb_defect_submission_bundle.zip"
with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as z:
    for pt in Path("pcb_competition").rglob("best.pt"):
        z.write(pt, arcname=f"weights/{pt.parent.parent.name}_best.pt")
    for pth in Path("pcb_runs").rglob("*.pth"):
        z.write(pth, arcname=f"weights/{pth.name}")
    for csv_file in [
        "submission.csv",
        "submission_ensemble.csv",
        "submission_yolo11x.csv",
        "submission_tta.csv",
        "submission_yolov8x.csv",
        "submission_rtdetr.csv",
        "submission_fasterrcnn_resnet50.csv",
        "submission_fasterrcnn_mobilenet.csv",
    ]:
        if Path(csv_file).exists():
            z.write(csv_file, arcname=csv_file)
    if Path("dataset_colab.yaml").exists():
        z.write("dataset_colab.yaml", arcname="dataset.yaml")

print(f"✅ Created comprehensive bundle: {zip_filename}")

# Automatic Colab download prompt
try:
    from google.colab import files
    files.download(zip_filename)
    print("⬇️ Download started for submission bundle.")
except ImportError:
    print(f"Local environment: bundle ready at {zip_filename}")
